In [1]:
import pandas as pd
import numpy as np

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
airports = pd.read_csv("../../data/raw/airports/airports.csv")

In [3]:
airports_clean = airports.copy()

In [4]:
missing = pd.DataFrame({
    "Missing Values": airports_clean.isnull().sum(),
    "Percentage": (airports_clean.isnull().mean() * 100).round(2)
})

missing.sort_values(by="Percentage", ascending=False)

,Missing Values,Percentage
home_link,80932,94.47
iata_code,76614,89.43
icao_code,75451,88.07
wikipedia_link,68948,80.48
keywords,64194,74.93
local_code,49719,58.04
gps_code,41334,48.25
continent,39649,46.28
elevation_ft,14861,17.35
municipality,4706,5.49


In [5]:
columns_to_drop = [
    "home_link",
    "wikipedia_link",
    "keywords",
    "local_code"
]

airports_clean.drop(columns=columns_to_drop, inplace=True)

In [6]:
airports_clean["elevation_ft"] = airports_clean["elevation_ft"].fillna(
    airports_clean["elevation_ft"].median()
)

In [7]:
airports_clean["elevation_ft"].isnull().sum()

np.int64(0)

In [8]:
airports_clean["municipality"] = airports_clean["municipality"].fillna("Unknown")

In [9]:
airports_clean["municipality"].isnull().sum()

np.int64(0)

In [10]:
airports_clean["iso_country"] = airports_clean["iso_country"].fillna("Unknown")

In [11]:
airports_clean["continent"] = airports_clean["continent"].fillna("Unknown")

In [12]:
missing_after = pd.DataFrame({
    "Missing Values": airports_clean.isnull().sum(),
    "Percentage": (
        airports_clean.isnull().mean() * 100
    ).round(2)
})

missing_after.sort_values(
    by="Percentage",
    ascending=False
)

,Missing Values,Percentage
iata_code,76614,89.43
icao_code,75451,88.07
gps_code,41334,48.25
name,0,0.00
id,0,0.00
ident,0,0.00
type,0,0.00
elevation_ft,0,0.00
longitude_deg,0,0.00
latitude_deg,0,0.00


In [13]:
text_columns = airports_clean.select_dtypes(include="object").columns

for col in text_columns:
    airports_clean[col] = airports_clean[col].str.strip()

In [14]:
code_columns = [
    "ident",
    "iso_country",
    "iso_region",
    "iata_code",
    "icao_code",
    "gps_code"
]

for col in code_columns:
    airports_clean[col] = airports_clean[col].str.upper()

In [17]:
airports_clean["scheduled_service"] = (
    airports_clean["scheduled_service"]
    .str.lower()
)


In [18]:
print("Duplicate Rows:", airports_clean.duplicated().sum())

Duplicate Rows: 0


In [19]:
print("Duplicate Airport Identifiers:", airports_clean["ident"].duplicated().sum())

Duplicate Airport Identifiers: 0


In [20]:
invalid_latitude = airports_clean[
    (airports_clean["latitude_deg"] < -90) |
    (airports_clean["latitude_deg"] > 90)
]

print("Invalid Latitude Records:", len(invalid_latitude))

Invalid Latitude Records: 0


In [21]:
invalid_longitude = airports_clean[
    (airports_clean["longitude_deg"] < -180) |
    (airports_clean["longitude_deg"] > 180)
]

print("Invalid Longitude Records:", len(invalid_longitude))

Invalid Longitude Records: 0


In [22]:
airports_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85670 entries, 0 to 85669
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 85670 non-null  int64  
 1   ident              85670 non-null  object 
 2   type               85670 non-null  object 
 3   name               85670 non-null  object 
 4   latitude_deg       85670 non-null  float64
 5   longitude_deg      85670 non-null  float64
 6   elevation_ft       85670 non-null  float64
 7   continent          85670 non-null  object 
 8   iso_country        85670 non-null  object 
 9   iso_region         85670 non-null  object 
 10  municipality       85670 non-null  object 
 11  scheduled_service  85670 non-null  object 
 12  icao_code          10219 non-null  object 
 13  iata_code          9056 non-null   object 
 14  gps_code           44336 non-null  object 
dtypes: float64(3), int64(1), object(11)
memory usage: 9.8+ MB


In [23]:
airports_clean.to_csv(
    "../../data/processed/airports_processed.csv",
    index=False
)

print("Airport dataset cleaned and saved successfully.")

Airport dataset cleaned and saved successfully.
